In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Our original forecasting dataset
forecast = pd.read_csv(
    project_dir / "data/processed/pm25_forecasting.csv",
    parse_dates=["date", "target_date"]
)

# Daily historical weather data
weather = pd.read_csv(
    project_dir / "data/processed/statewide_weather_daily.csv",
    parse_dates=["date"]
)

print("Forecasting records:", len(forecast))
print("Weather records:", len(weather))


Forecasting records: 8283
Weather records: 8924


In [2]:

weather_columns = [
    "station_id",
    "date",
    "temperature_mean",
    "humidity_mean",
    "wind_speed_mean",
    "precipitation_total",
    "wind_sin_mean",
    "wind_cos_mean"
]

combined_weather = forecast.merge(
    weather[weather_columns],
    on=["station_id", "date"],
    how="left",
    validate="one_to_one"
)

print("Merged observations:", len(combined_weather))

print("\nMissing weather data:")
print(
    combined_weather[
        weather_columns[2:]
    ].isna().sum()
)

display(combined_weather.head())


Merged observations: 8283

Missing weather data:
temperature_mean       0
humidity_mean          0
wind_speed_mean        0
precipitation_total    0
wind_sin_mean          0
wind_cos_mean          0
dtype: int64


,State Code,County Code,Site Num,Parameter Code,POC,Latitude,Longitude,Datum,Parameter Name,Sample Duration,...,frp_sum_100km,pm25_original,target_pm25,target_date,temperature_mean,humidity_mean,wind_speed_mean,precipitation_total,wind_sin_mean,wind_cos_mean
0,6,7,8,88101,3,39.76168,-121.84047,NAD83,PM2.5 - Local Conditions,24-HR BLK AVG,...,55.47,9.1,8.8,2025-07-02,31.308333,28.083333,9.545833,0.0,0.139949,-0.240726
1,6,7,8,88101,3,39.76168,-121.84047,NAD83,PM2.5 - Local Conditions,24-HR BLK AVG,...,30.24,8.8,10.0,2025-07-03,31.233333,29.750000,7.625000,0.0,0.283526,-0.347032
2,6,7,8,88101,3,39.76168,-121.84047,NAD83,PM2.5 - Local Conditions,24-HR BLK AVG,...,22.92,10.0,6.7,2025-07-04,28.366667,35.916667,12.516667,0.0,0.082590,-0.668494
3,6,7,8,88101,3,39.76168,-121.84047,NAD83,PM2.5 - Local Conditions,24-HR BLK AVG,...,17.04,6.7,13.2,2025-07-05,25.512500,32.375000,15.720833,0.0,0.537723,-0.817414
4,6,7,8,88101,3,39.76168,-121.84047,NAD83,PM2.5 - Local Conditions,24-HR BLK AVG,...,27.69,13.2,6.1,2025-07-06,25.183333,28.666667,7.237500,0.0,-0.072326,-0.524710


In [3]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

base_features = ["pm25"]

fire_features = [
    "pm25",
    "fire_count_50km",
    "fire_count_100km",
    "frp_sum_100km"
]

weather_features = [
    "temperature_mean",
    "humidity_mean",
    "wind_speed_mean",
    "precipitation_total",
    "wind_sin_mean",
    "wind_cos_mean"
]

all_features = fire_features + weather_features

# Remove incomplete observations, if any
data = combined_weather.dropna(
    subset=all_features + ["target_pm25"]
).copy()

# Split by the date being predicted
train = data[data["target_date"] < "2025-09-01"]
test = data[data["target_date"] >= "2025-09-01"]

print("Training records:", len(train))
print("Testing records:", len(test))

y_train = train["target_pm25"]
y_test = test["target_pm25"]

def fit_predict(features):
    model = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(train[features], y_train)
    return model, model.predict(test[features])

# Baseline
pred_persistence = test["pm25"].to_numpy()

# PM2.5 model
rf_base, pred_base = fit_predict(base_features)

# Fire model
rf_fire, pred_fire = fit_predict(fire_features)

# Combined fire + weather model
rf_weather, pred_weather = fit_predict(all_features)

def evaluate(name, preds):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, preds)
        )
    }

results_weather = pd.DataFrame([
    evaluate("Persistence", pred_persistence),
    evaluate("RF - PM2.5 Only", pred_base),
    evaluate("RF - PM2.5 + Fire", pred_fire),
    evaluate("RF - Fire + Weather", pred_weather)
])

display(results_weather.round(3))


Training records: 5558
Testing records: 2725


,Model,MAE,RMSE
0,Persistence,2.577,11.482
1,RF - PM2.5 Only,2.422,11.383
2,RF - PM2.5 + Fire,2.485,11.449
3,RF - Fire + Weather,2.411,11.344


In [4]:

# Evaluate high-PM2.5 observations
high_mask = y_test >= 35

print(
    "Elevated PM2.5 observations:",
    high_mask.sum()
)

if high_mask.sum() > 0:
    extreme_results = pd.DataFrame([
        {
            "Model": "RF - PM2.5 Only",
            "Elevated MAE": mean_absolute_error(
                y_test[high_mask],
                pred_base[high_mask.to_numpy()]
            )
        },
        {
            "Model": "RF - PM2.5 + Fire",
            "Elevated MAE": mean_absolute_error(
                y_test[high_mask],
                pred_fire[high_mask.to_numpy()]
            )
        },
        {
            "Model": "RF - Fire + Weather",
            "Elevated MAE": mean_absolute_error(
                y_test[high_mask],
                pred_weather[high_mask.to_numpy()]
            )
        }
    ])

    display(extreme_results.round(3))


Elevated PM2.5 observations: 17


,Model,Elevated MAE
0,RF - PM2.5 Only,98.166
1,RF - PM2.5 + Fire,97.611
2,RF - Fire + Weather,96.468


In [5]:

# Weather features without wildfire variables
pm_weather_features = [
    "pm25"
] + weather_features

# Reuse the same training and testing observations
rf_weather_only, pred_weather_only = fit_predict(
    pm_weather_features
)

# Compare all models
results_final = pd.DataFrame([
    evaluate("Persistence", pred_persistence),
    evaluate("RF - PM2.5 Only", pred_base),
    evaluate("RF - PM2.5 + Fire", pred_fire),
    evaluate("RF - PM2.5 + Weather", pred_weather_only),
    evaluate("RF - Fire + Weather", pred_weather)
])

display(
    results_final.sort_values("MAE").round(4)
)


,Model,MAE,RMSE
4,RF - Fire + Weather,2.4107,11.3435
1,RF - PM2.5 Only,2.4224,11.3825
3,RF - PM2.5 + Weather,2.4232,11.3334
2,RF - PM2.5 + Fire,2.4845,11.4488
0,Persistence,2.5774,11.4819


In [6]:

prediction_comparison = test[
    [
        "station_id",
        "date",
        "target_date",
        "pm25",
        "target_pm25"
    ]
].copy()

prediction_comparison["PM2.5 Only"] = pred_base
prediction_comparison["PM2.5 + Fire"] = pred_fire
prediction_comparison["PM2.5 + Weather"] = pred_weather_only
prediction_comparison["Fire + Weather"] = pred_weather

# Inspect September 8 pollution predictions
event_predictions = prediction_comparison[
    (
        prediction_comparison["target_date"]
        == "2025-09-08"
    )
    & (
        prediction_comparison["station_id"].isin([
            "06-051-0005",
            "06-027-0002",
            "06-027-1033"
        ])
    )
]

display(event_predictions.round(3))


/var/folders/xf/_sn5vxxj05nfsdm_3mlrbfz00000gn/T/ipykernel_96006/3841114979.py:31: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(event_predictions.round(3))


,station_id,date,target_date,pm25,target_pm25,PM2.5 Only,PM2.5 + Fire,PM2.5 + Weather,Fire + Weather
1254,06-027-0002,2025-09-07,2025-09-08,189.8,280.5,29.853,30.266,32.716,33.280
1554,06-027-1033,2025-09-07,2025-09-08,98.3,204.4,29.853,30.816,21.604,25.719
2875,06-051-0005,2025-09-07,2025-09-08,139.9,427.9,29.853,24.744,36.653,31.182
